# LC 39 — Combination Sum
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Backtracking
**Pattern:** Backtrack with Repetition — Reuse Same Index

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Each candidate can
be used unlimited times. Pass the same index i
(not i+1) when recursing to allow reuse. Prune
when the remaining target goes negative.
</div>

## Official Problem Statement

Given an array of **distinct** integers `candidates`
and a target integer `target`, return a list of all
**unique combinations** of `candidates` where the
chosen numbers sum to `target`.

You may return the combinations in any order. The
**same** number may be chosen from `candidates`
an **unlimited number of times**. Two combinations
are unique if the frequency of at least one of
the chosen numbers is different.

**Example 1:**
```
Input:  candidates = [2,3,6,7], target = 7
Output: [[2,2,3],[7]]
```
**Example 2:**
```
Input:  candidates = [2,3,5], target = 8
Output: [[2,2,2,2],[2,3,3],[3,5]]
```

**Constraints:**
- `1 <= candidates.length <= 30`
- `2 <= candidates[i] <= 40`
- All elements are distinct
- `1 <= target <= 40`

## What This Is Actually Asking

Pick numbers from the list (repeats allowed) that
add up to the target. Return every unique way to
do it. Order within a combination does not matter —
[2,2,3] and [3,2,2] are the same combination.

## Walk Through an Example by Hand

```
candidates=[2,3,6,7]  target=7

backtrack(start=0, path=[], remain=7):

  i=0 pick 2: path=[2] remain=5
    i=0 pick 2: path=[2,2] remain=3
      i=0 pick 2: path=[2,2,2] remain=1
        i=0 pick 2: remain=-1 < 0 -> PRUNE
        i=1 pick 3: remain=-2 < 0 -> PRUNE
      pop 2: path=[2,2]
      i=1 pick 3: path=[2,2,3] remain=0 -> ADD!
      pop 3: path=[2,2]
      i=2 pick 6: remain=-3 -> PRUNE
    pop 2: path=[2]
    i=1 pick 3: path=[2,3] remain=2
      i=1 pick 3: remain=-1 -> PRUNE
      (2<3 already, 2 only reachable via i=0)
    pop 3
    ... (6 and 7 exceed 5 — pruned)
  pop 2: path=[]

  i=3 pick 7: path=[7] remain=0 -> ADD!

Result: [[2,2,3],[7]]
```

## The Picture

```
candidates=[2,3,6,7]  target=7

Decision tree (showing start index):

  remain=7
  |
  pick 2 (reuse i=0 allowed)
  remain=5
  |
  pick 2  remain=3
  |
  pick 2  remain=1  -> all candidates > 1 -> prune
  pick 3  remain=0  -> FOUND [2,2,3]

Key difference from Subsets:

  Subsets:    recurse(i + 1)   no reuse
  Comb Sum:   recurse(i)       reuse same element

  Using start index prevents [3,2,2] after [2,2,3]
  (only move forward in the array, never backward)

Prune early:
  if remain < 0: return  (over-shot the target)
  if remain == 0: add path[:]; return  (exact hit)
```

## When To Use This Pattern

- When elements can be **reused**, think
  **recurse with same index i, not i+1**
- When remain hits 0, think
  **valid combination — add path copy to result**
- When remain goes negative, think
  **prune — return immediately**
- When candidates are sorted, think
  **break inner loop early when candidate > remain**

## The Approach

Sort the candidates for early pruning. Use a
recursive helper with start index, current path,
and remaining target. At each level, try every
candidate from start onwards. If the candidate
equals the remainder, save the path. If less,
add it and recurse using the same index. If
greater, break (sorted order). Undo after each
recursive call.

In [2]:
from typing import List  # type hints for the solution

In [3]:
def test_harness(func):
    def norm(res):
        return sorted(tuple(sorted(c)) for c in res)

    tests = [
        # (candidates, target, expected)
        ([2,3,6,7],  7, [[2,2,3],[7]]),
        ([2,3,5],    8, [[2,2,2,2],[2,3,3],[3,5]]),
        ([2],        1, []),         # impossible
        ([1],        1, [[1]]),
        ([1],        2, [[1,1]]),
        ([2,7,6,3],  7, [[2,2,3],[7]]),  # unsorted input
    ]

    passed = 0
    for i, (cands, tgt, expected) in enumerate(tests):
        result = func(cands[:], tgt)
        ok = norm(result) == norm(expected)
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"target={tgt} | "
            f"expected={sorted(expected)} | "
            f"got={sorted(result)}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [6]:
def combinationSum(
    candidates: List[int], target: int
) -> List[List[int]]:
    result = []

    def backtrack(start , current, remaining):
        if remaining == 0:   # A valid combo found
            result.append(current[:])
            return
        if remaining < 0:    # went so far
            return
        for i in range(start, len(candidates)):
            current.append(candidates[i])
            backtrack(i,current, remaining - candidates[i])
            current.pop()
    backtrack( 0 , [] , target)

    return result





#Quick debug — run this cell while building
print(combinationSum([2,3,6,7], 7))   # [[2,2,3],[7]]
print(combinationSum([2,3,5], 8))     # [[2,2,2,2],[2,3,3],[3,5]]
print(combinationSum([2], 1))          # []
print(combinationSum([1], 2))          # [[1,1]]
test_harness(combinationSum)    


    

[[2, 2, 3], [7]]
[[2, 2, 2, 2], [2, 3, 3], [3, 5]]
[]
[[1, 1]]
Test 1: PASSED | target=7 | expected=[[2, 2, 3], [7]] | got=[[2, 2, 3], [7]]
Test 2: PASSED | target=8 | expected=[[2, 2, 2, 2], [2, 3, 3], [3, 5]] | got=[[2, 2, 2, 2], [2, 3, 3], [3, 5]]
Test 3: PASSED | target=1 | expected=[] | got=[]
Test 4: PASSED | target=1 | expected=[[1]] | got=[[1]]
Test 5: PASSED | target=2 | expected=[[1, 1]] | got=[[1, 1]]
Test 6: PASSED | target=7 | expected=[[2, 2, 3], [7]] | got=[[2, 2, 3], [7]]

6/6 tests passed


In [7]:
def combinationSum(
    candidates: List[int], target: int
) -> List[List[int]]:
    """
    Return all combinations summing to target (reuse ok).

    Sort candidates. Backtrack with start index and
    remain. Loop from start: if candidates[i] > remain
    break; if == remain add path copy; else recurse
    with same i (allow reuse). Pop after each call.

    Time:  O(n^(t/m)) where t=target, m=min candidate
    Space: O(t/m) — max recursion depth
    """
    result = []

    def backtrack(start, current, remaining):
        if remaining == 0:        # found a valid combo
            result.append(current[:])
            return
        if remaining < 0:         # overshot — prune
            return
        for i in range(start, len(candidates)):
            current.append(candidates[i])
            backtrack(i, current, remaining - candidates[i])  # i not i+1 — reuse allowed
            current.pop()         # undo — try next candidate
            
    backtrack(0, [], target)
    return result

        
# Quick debug — run this cell while building
print(combinationSum([2,3,6,7], 7))   # [[2,2,3],[7]]
print(combinationSum([2,3,5], 8))     # [[2,2,2,2],[2,3,3],[3,5]]
print(combinationSum([2], 1))          # []
print(combinationSum([1], 2))          # [[1,1]]
test_harness(combinationSum)

[[2, 2, 3], [7]]
[[2, 2, 2, 2], [2, 3, 3], [3, 5]]
[]
[[1, 1]]
Test 1: PASSED | target=7 | expected=[[2, 2, 3], [7]] | got=[[2, 2, 3], [7]]
Test 2: PASSED | target=8 | expected=[[2, 2, 2, 2], [2, 3, 3], [3, 5]] | got=[[2, 2, 2, 2], [2, 3, 3], [3, 5]]
Test 3: PASSED | target=1 | expected=[] | got=[]
Test 4: PASSED | target=1 | expected=[[1]] | got=[[1]]
Test 5: PASSED | target=2 | expected=[[1, 1]] | got=[[1, 1]]
Test 6: PASSED | target=7 | expected=[[2, 2, 3], [7]] | got=[[2, 2, 3], [7]]

6/6 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(combinationSum)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force all permutations | O(n^target) | O(target) |
| Backtrack with pruning | O(n^(t/m)) | O(t/m) |

Pruning (break when candidate > remain) cuts the
tree significantly. Sorting candidates first
ensures the break is hit as early as possible.

## Real World Connection

At Citi, the budget allocation system finds all
combinations of infrastructure components whose
costs sum exactly to the approved spend target.
Components can be provisioned multiple times
(e.g., additional Lambda memory tiers), so reuse
is allowed — exactly Combination Sum.
The backtracking prune cuts the search early when
a partial allocation already exceeds the budget,
making the enumeration feasible for dozens of
component types.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra